In [2]:
# =========================
# 02 MODEL TRAINING
# Serie A Match Prediction
# =========================

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/Serie_A_Forecast"
# =========================
# LOAD PROCESSED DATASET
# =========================

df_final = pd.read_csv(
    f"{PROJECT_PATH}/data/processed/df_final.csv"
)

#df_final = pd.read_csv("df_final.csv")

df_final.head()

Mounted at /content/drive


,Date,HomeTeam,AwayTeam,FTR,home_avg_goals_5,away_avg_goals_5,home_avg_goals_conceded_5,away_avg_goals_conceded_5,home_avg_points_5,away_avg_points_5,home_avg_shots_5,away_avg_shots_5,home_avg_shots_target_5,away_avg_shots_target_5,home_home_points_5,away_away_points_5,home_home_goals_5,away_away_goals_5
0,2025-09-13,Cagliari,Parma,H,0.5,0.5,1.0,1.5,0.5,0.5,12.0,7.5,3.5,2.5,1.0,0.0,1.0,0.0
1,2025-09-14,Roma,Torino,A,1.0,0.0,0.0,2.5,3.0,0.5,12.0,9.5,4.5,4.0,3.0,0.0,1.0,0.0
2,2025-09-14,Atalanta,Lecce,H,1.0,0.0,1.0,1.0,1.0,0.5,16.0,7.0,4.5,1.0,1.0,1.0,1.0,0.0
3,2025-09-14,Pisa,Udinese,A,0.5,1.5,1.0,1.0,0.5,2.0,9.0,12.0,2.5,4.5,0.0,3.0,0.0,2.0
4,2025-09-14,Sassuolo,Lazio,H,1.0,2.0,2.5,1.0,0.0,1.5,8.0,13.5,4.0,3.5,0.0,0.0,0.0,0.0


In [3]:
# =========================
# BASIC CHECKS
# =========================

print(df_final.shape)
print(df_final.isna().sum())
print(df_final["FTR"].value_counts())

(324, 18)
Date                         0
HomeTeam                     0
AwayTeam                     0
FTR                          0
home_avg_goals_5             0
away_avg_goals_5             0
home_avg_goals_conceded_5    0
away_avg_goals_conceded_5    0
home_avg_points_5            0
away_avg_points_5            0
home_avg_shots_5             0
away_avg_shots_5             0
home_avg_shots_target_5      0
away_avg_shots_target_5      0
home_home_points_5           0
away_away_points_5           0
home_home_goals_5            0
away_away_goals_5            0
dtype: int64
FTR
H    129
A    109
D     86
Name: count, dtype: int64


In [4]:
# =========================
# CREATE X AND y
# =========================

X = df_final.drop(
    columns=["Date", "HomeTeam", "AwayTeam", "FTR"]
)

y = df_final["FTR"]

print(X.head())
print(y.head())

   home_avg_goals_5  away_avg_goals_5  home_avg_goals_conceded_5  \
0               0.5               0.5                        1.0   
1               1.0               0.0                        0.0   
2               1.0               0.0                        1.0   
3               0.5               1.5                        1.0   
4               1.0               2.0                        2.5   

   away_avg_goals_conceded_5  home_avg_points_5  away_avg_points_5  \
0                        1.5                0.5                0.5   
1                        2.5                3.0                0.5   
2                        1.0                1.0                0.5   
3                        1.0                0.5                2.0   
4                        1.0                0.0                1.5   

   home_avg_shots_5  away_avg_shots_5  home_avg_shots_target_5  \
0              12.0               7.5                      3.5   
1              12.0               9.5 

In [5]:
# =========================
# TEMPORAL TRAIN / TEST SPLIT
# =========================

split = int(len(df_final) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (259, 14)
Test size: (65, 14)


In [6]:
# =========================
# TRAIN LOGISTIC REGRESSION MODEL
# =========================

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [7]:
# =========================
# MAKE PREDICTIONS
# =========================

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.4461538461538462


In [8]:
# =========================
# CLASSIFICATION REPORT
# =========================

print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           A       0.57      0.62      0.59        21
           D       0.29      0.39      0.33        18
           H       0.50      0.35      0.41        26

    accuracy                           0.45        65
   macro avg       0.45      0.45      0.44        65
weighted avg       0.46      0.45      0.45        65



In [9]:
# =========================
# CONFUSION MATRIX
# =========================

cm = confusion_matrix(y_test, predictions, labels=model.classes_)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{c}" for c in model.classes_],
    columns=[f"Pred_{c}" for c in model.classes_]
)

cm_df

,Pred_A,Pred_D,Pred_H
Real_A,13,4,4
Real_D,6,7,5
Real_H,4,13,9


In [10]:
# =========================
# SAVE PREDICTIONS
# =========================

df_predictions = df_final.iloc[split:].copy()

df_predictions["Predicted_Result"] = predictions

probabilities = model.predict_proba(X_test)

prob_df = pd.DataFrame(
    probabilities,
    columns=[f"Prob_{c}" for c in model.classes_]
)

df_predictions = pd.concat(
    [df_predictions.reset_index(drop=True), prob_df.reset_index(drop=True)],
    axis=1
)

df_predictions.head()

,Date,HomeTeam,AwayTeam,FTR,home_avg_goals_5,away_avg_goals_5,home_avg_goals_conceded_5,away_avg_goals_conceded_5,home_avg_points_5,away_avg_points_5,...,home_avg_shots_target_5,away_avg_shots_target_5,home_home_points_5,away_away_points_5,home_home_goals_5,away_away_goals_5,Predicted_Result,Prob_A,Prob_D,Prob_H
0,2026-03-14,Inter,Atalanta,D,2.4,1.8,0.6,1.2,2.4,2.0,...,7.4,6.4,2.6,1.6,2.8,1.2,H,0.227400,0.338307,0.434293
1,2026-03-15,Verona,Genoa,A,0.8,1.4,1.6,1.2,0.8,1.4,...,1.8,4.2,0.2,0.6,0.8,0.6,A,0.595469,0.269809,0.134722
2,2026-03-15,Pisa,Cagliari,H,0.2,0.4,1.6,1.4,0.2,0.4,...,2.2,2.2,0.2,1.0,0.6,1.0,D,0.334252,0.409035,0.256713
3,2026-03-15,Sassuolo,Bologna,A,1.6,1.0,1.8,0.8,1.8,1.8,...,3.6,3.4,1.8,2.0,1.2,1.8,A,0.480086,0.370264,0.149650
4,2026-03-15,Como,Roma,H,1.8,2.2,1.0,1.4,2.0,1.6,...,3.6,3.6,1.4,1.4,2.2,1.4,D,0.308107,0.504886,0.187007


In [11]:
# =========================
# SAVE MODEL AND OUTPUTS
# =========================


joblib.dump(
    model,
    f"{PROJECT_PATH}/models/logistic_model.pkl"
)

#joblib.dump(model, "logistic_model.pkl")

df_predictions.to_csv(
    f"{PROJECT_PATH}/outputs/predictions.csv",
    index=False
)

#df_predictions.to_csv("predictions.csv", index=False)

cm_df.to_csv("confusion_matrix.csv")

print("Model and outputs saved successfully.")

Model and outputs saved successfully.


In [12]:
# =========================
# FEATURE COEFFICIENTS
# =========================

coef_df = pd.DataFrame(
    model.coef_,
    columns=X.columns,
    index=model.classes_
)

coef_df

,home_avg_goals_5,away_avg_goals_5,home_avg_goals_conceded_5,away_avg_goals_conceded_5,home_avg_points_5,away_avg_points_5,home_avg_shots_5,away_avg_shots_5,home_avg_shots_target_5,away_avg_shots_target_5,home_home_points_5,away_away_points_5,home_home_goals_5,away_away_goals_5
A,-0.116261,-0.201286,0.186938,0.063612,0.345903,0.443108,-0.037982,0.076908,-0.077304,0.060598,-0.437462,-0.227190,-0.115850,0.294455
D,-0.135068,0.204063,-0.157905,0.076645,-0.118701,0.004557,-0.063043,-0.004044,0.040913,-0.124898,0.174110,0.074926,0.250417,-0.073195
H,0.251329,-0.002777,-0.029034,-0.140257,-0.227202,-0.447665,0.101026,-0.072864,0.036391,0.064300,0.263353,0.152264,-0.134567,-0.221260
